# CVRIE - Unsupervised Learning: Patient Testimonies Clustering

## 1. Introduction and context

We cluster 1,011 unlabelled patient testimonies by condition to speed up hospital admission.
Each row holds an id, a free-text testimony, and sometimes a colour code; the colour is never a
model input and serves only to colour the final plot. There are no labels, so we judge quality
with internal metrics and by reading the testimonies inside each cluster. The pipeline is: load,
EDA, normalise and filter with spaCy, TF-IDF, LSA, clustering, evaluation.

## 2. Loading the data

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
from pathlib import Path
import pandas as pd
from src.unsupervised.loader import read_testimonies

RAW = Path("../data/raw/unsupervised/Student_Dataset.csv")
frame = read_testimonies(RAW)

## 3. Exploratory Data Analysis (EDA)

### 3.1 Prove the naive parse is wrong

In [ ]:
naive = pd.read_csv(RAW, header=None, names=["id", "color", "text"], encoding="utf-8")
correct = read_testimonies(RAW)

print("Naive shape:", naive.shape)
print("Correct shape:", correct.shape)

print(naive.head())
print(correct.head())

print(f"Naive rows={len(naive)}, text is null:{naive["text"].isnull().sum()}")
print(
    f"Correct rows={len(correct)}, text is null:{correct["testimony"].isnull().sum()}"
)

broken = naive[naive["text"].isnull()].iloc[0]
print(broken.color[:70], "|", broken.text)

`pd.read_csv` returns 1,011 rows and raises no error, but 397 testimonies (39.3%) land in the
colour column and leave the text column empty. The file has no header and carries 2 or 3 fields
per line, so pandas fills the three names left to right and a 2-field line puts the testimony
where the colour belongs. `read_testimonies` assigns fields by counting them per line instead,
which leaves 0 missing testimonies.

### 3.2 Shape and is the missing column random

In [ ]:
correct["number_words"] = correct["testimony"].str.split().str.len()
print(
    correct.groupby("color", dropna=False)["number_words"]
    .agg(["count", "mean", "min", "max"])
    .round(1)
)

In [ ]:
correct.groupby(pd.cut(correct["id"], bins=5), observed=True)["color_present"].agg(
    ["size", "mean"]
)

The colour is present on 614 rows and absent on 397. Mean length is 28.6 words with the colour and 26.8 without, and the share of rows carrying a colour stays between 0.584 and 0.639 across five equal id ranges. The absence looks random, with no link to position in the file and only a small difference in length. The colour is never a model input.

### 3.3 Duplicates

In [ ]:
duplicate = correct[correct.duplicated("testimony", keep=False)]
print(f"distinct text: {correct["testimony"].nunique()} / {len(correct)}")
print(
    f"duplicate row : {len(duplicate)} ({duplicate["testimony"].nunique()} distinct texts)"
)
print(duplicate.sort_values("testimony")[["id", "color", "testimony"]])

summary = (
    duplicate.groupby("testimony")
    .agg(copies=("id", "size"), ids=("id", list), with_colour=("color_present", "sum"))
    .sort_values("copies", ascending=False)
    .reset_index()
)
print(summary[["copies", "ids", "with_colour", "testimony"]].to_string(max_colwidth=55))

9 texts appear twice, which leaves 1,002 distinct testimonies out of 1,011. Every duplicate is an exact pair, never a triple. We keep them because each patient still needs a cluster assignment, and 9 extra points out of 1,011 are too few to shift a centroid.

### 3.4 Length distribution

In [ ]:
correct.number_words.describe()

In [ ]:
correct.number_words.hist(bins=20)

Testimonies run from 13 to 55 words, with a median of 27 and a mean of 27.9. The longest holds
more than four times the words of the shortest, so long testimonies repeat words more often and
would dominate raw term counts. We apply sublinear term frequency to damp repeated terms and L2
normalisation so that length does not drive similarity.

### 3.5 Non-ASCII Inventory

In [ ]:
import collections
import unicodedata

non_ascii_counts = collections.Counter(
    character
    for testimony in correct["testimony"]
    for character in testimony
    if ord(character) > 127
)

for character, count in non_ascii_counts.most_common():
    name = unicodedata.name(character, "<no name>")
    print(f"{character!r:10} U+{ord(character):04X} {count:<5} {name}")

Six non-ASCII characters appear: the right single quotation mark 461 times, the non-breaking hyphen 112 times, the degree sign and the em dash 6 times each, and the left single quotation mark and the narrow no-break space twice each. They render like ordinary apostrophes, hyphens and spaces but carry different codepoints, so a tokeniser would treat `don’t` and `don't` as different words. We fold quotes and hyphens to ASCII and special spaces to a normal space before spaCy. The em dash always joins two words without spaces (`dreams—talking`), so we fold it to a space rather than a hyphen. The degree sign only appears in body temperatures next to the word *fever* (`37.5°C`), so we fold it to a space and let the number be filtered out: *fever* already carries the signal.

### 3.6 Raw word peek before any preprocessing

In [ ]:
word_counts = collections.Counter(
    word.strip('.?!,"').lower()
    for testimony in correct["testimony"]
    for word in testimony.split()
)
vocabulary_size = len(word_counts)
hapax_count = sum(1 for count in word_counts.values() if count == 1)

doc_freq = collections.Counter(
    word
    for testimony in correct["testimony"]
    for word in {w.strip('.?!,"').lower() for w in testimony.split()}
)
single_doc = sum(1 for count in doc_freq.values() if count == 1)

print(f"words in exactly one testimony: {single_doc}")
print(f"total tokens: {sum(word_counts.values())} | distinct: {vocabulary_size}")
print(word_counts.most_common(15))
print(f"vocab: {vocabulary_size} | words appearing once in the corpus: {hapax_count}")

The corpus holds 28,189 tokens over a vocabulary of 2,470 words. The 15 most frequent are function words (`a`, `i`, `the`, `my`, `and`), with `feel` the first content word at rank 12. 964 words appear exactly once in the corpus (39.0% of the vocabulary), and 968 appear in a single testimony. We remove stop words, since the top of the ranking carries no clinical signal, and we set a `min_df` floor, since a word present in only one testimony cannot make two testimonies similar.

### 3.7 Preprocessing decisions

- Parse the raw file by field count rather than with `pd.read_csv`. (3.1)
- Keep the colour out of every model input and use it only for the final plot. (3.2)
- Keep the 9 duplicated testimonies. (3.3)
- Use sublinear term frequency and L2 normalisation in TF-IDF. (3.4)
- Fold quotes and hyphens to ASCII, and special spaces, em dashes and degree signs to a space, before spaCy. (3.5)
- Remove stop words. (3.6)
- Set a `min_df` floor, with the value chosen after measuring on the preprocessed tokens. (3.6)

## 4. Preprocessing

In [ ]:
from src.unsupervised.preprocess import analyze, analyze_corpus

tokens = analyze_corpus(correct["testimony"].tolist())
assert tokens[5] == analyze(correct["testimony"].iloc[5])

vocabulary = set()
for doc_tokens in tokens:
    vocabulary.update(doc_tokens)

print(tokens)
print("testimonies:", len(tokens))
print("vocabulary after preprocessing:", len(vocabulary))

print(correct["testimony"].iloc[1])
print(tokens[1])

In [ ]:
from spacy import displacy
from src.unsupervised.preprocess import nlp

my_text = correct["testimony"].iloc[1]
print(my_text)

doc = nlp(my_text)
print(f"\n{'token':12} {'lemma':12} {'pos':6} stop")
for token in doc:
    print(token.text, token.lemma_, token.pos_, token.is_stop)

options = {"compact": True, "color": "#329ED9", "bg": "#D6E6F6", "font": "Arial"}

displacy.render(doc, style="dep", jupyter=True, options=options)

`preprocess.py` folds typographic unicode to ASCII, then hands each testimony to spaCy and keeps
the lowercase lemma of every token that is not punctuation, a number, whitespace or a stop word,
and whose part of speech is a noun, adjective or verb. The vocabulary drops from 2,470 raw words
to 1513.

- **Lemma, not lowercase**: merges forms such as *feel* / *feels* / *feeling* into one dimension.
  spaCy's lemmatizer is rule-based and needs the part-of-speech tag, which is why the tagger runs
  before it.
- **Nouns, adjectives and verbs**: filtering by part of speech removes narration and function
  words by a rule rather than by a hand-written list. Keeping verbs matters because many symptoms
  are verbs (*cough*, *itch*, *swell*).
- **Stop words**: we use spaCy's English stop list as is. It also removes some body-location
  words (*back*, *side*, *top*, *full*), which is a known limitation we may revisit.
- **Negation**: spaCy's stop list removes *not*, *no* and *n't*, so a negated symptom becomes
  indistinguishable from a present one. We document this as a limitation rather than handle it.

The dependency tree above shows what spaCy produces: a part of speech and a grammatical relation
for each token, plus the lemma used downstream.

## 5. Vectorisation

In [ ]:
from src.unsupervised.vectorize import build_vectorizer

vectorizer = build_vectorizer()
matrix = vectorizer.fit_transform(tokens)
features = vectorizer.get_feature_names_out()

density = matrix.nnz / (matrix.shape[0] * matrix.shape[1])
print("matrix:", matrix.shape)
print(f"non-zero cells: {matrix.nnz} ({density:.2%} dense)")
print("features kept:", len(features), "of", len(vocabulary), "vocabulary words")

TF-IDF over the preprocessed lemmas gives a 1,011 x [N] matrix that is [X]% dense: each testimony touches about 11 of the [N] possible terms. The matrix is stored in sparse form, since storing the zeros would waste 99% of the memory. This density is also the reason we reduce the dimensions before clustering: in a space this sparse,two testimonies rarely share any term at all, and distances between them carry little information.

In [ ]:
idf = pd.Series(vectorizer.idf_, index=features).sort_values()

print("lowest IDF (most common, least distinctive):")
print(idf.head(10))
print("\nhighest IDF (rarest, most distinctive):")
print(idf.tail(10))

IDF weights a term by how rare it is across testimonies, so the two ends of this ranking are a check on the preprocessing. The lowest-IDF terms are `feel`, `pain`, `sore`, `notice`, `ache`, `start`, `make` and `day`. `feel` appears in 49% of testimonies, nearly four times more often than the next term, and describes the act of reporting rather than the condition. The highest-IDF terms all sit at the `min_df` floor and are ordinary words rather than typos, which confirms the tokens reaching the vectorizer are clean.

In [ ]:
for value in [1, 2, 3, 5, 10]:
    test_matrix = build_vectorizer(min_df=value).fit_transform(tokens)
    print(
        f"min_df={value:2}  features={test_matrix.shape[1]:5}"
        f"  density={test_matrix.nnz / (test_matrix.shape[0] * test_matrix.shape[1]):.2%}"
    )

In [ ]:
strict = build_vectorizer(min_df=3)
strict.fit(tokens)

loose = build_vectorizer(min_df=2)
loose.fit(tokens)

dropped = set(loose.get_feature_names_out()) - set(strict.get_feature_names_out())
print(len(dropped), "terms dropped going from min_df=2 to 3:")
print(sorted(dropped)[:30])

`min_df` sets the minimum number of testimonies a term must appear in. Sweeping it gives 1,513 features at 1, 983 at 2, 750 at 3, 515 at 5 and 296 at 10. The first step removes 530 terms present in a single testimony, which by construction cannot make two testimonies similar. Each later step removes a steady 220 to 235 terms, so the sharp gain is between 1 and 2 and we stop at 3.

The 233 terms dropped between 2 and 3 are mostly incidental context (*backpack*, *breakfast*, *chore*, *beard*), though a few are clinical (*antibiotic*, *blurred*, *cholesterol*, *arch*). Each appears in exactly two testimonies, too few to shape a cluster of fifty or more, so we accept the loss. `max_df=0.5` removes no term here, since no word reaches half the corpus; we keep it as a guard rather than as an active filter.